# Time Series Analysis & Forecasting: Statistical Models to Tabular ML

Time series modeling addresses temporally correlated observations where the fundamental assumption of i.i.d. data is violated. This lab implements walk-forward temporal cross-validation, stationarity differencing, classical autoregressive modeling, and lag feature extraction for machine learning regression.

### Table of Contents
1. [Temporal Splitting & Avoiding Lookahead Leakage](#1-temporal-splitting--avoiding-lookahead-leakage)
2. [Stationarity & Order-1 Differencing](#2-stationarity--order-1-differencing)
3. [Autoregressive AR(p) Modeling & Multi-Step Forecasting](#3-autoregressive-arp-modeling--multi-step-forecasting)
4. [Tabular ML Feature Engineering (Lags, Rolling Stats, Cyclical Encodings)](#4-tabular-ml-feature-engineering-lags-rolling-stats-cyclical-encodings)
5. [Forecast Evaluation Metrics (RMSE vs. sMAPE)](#5-forecast-evaluation-metrics-rmse-vs-smape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid")
print("Loaded Time Series interactive lab environment.")

## 1. Temporal Splitting & Avoiding Lookahead Leakage
Standard cross-validation leaks future information to predict the past. Walk-Forward / Expanding Window validation preserves chronological causality.

In [ ]:
# Synthetic trend + seasonal time series
np.random.seed(42)
t = np.arange(120)
trend = 0.08 * t
seasonal = 3.0 * np.sin(2 * np.pi * t / 12.0)  # Period 12
noise = np.random.normal(0, 0.5, size=len(t))
series = 10.0 + trend + seasonal + noise

# Walk-Forward Split Visualization (4 folds)
plt.figure(figsize=(10, 4))
plt.plot(t, series, color="gray", alpha=0.5, label="Full Series")
split_point = 96
plt.plot(t[:split_point], series[:split_point], color="#1f77b4", lw=2, label="Training History (t <= 96)")
plt.plot(t[split_point:], series[split_point:], color="#d62728", lw=2, linestyle="--", label="Future Horizon (Test Set)")
plt.title("Strict Temporal Partitioning (No Lookahead Leakage)", fontweight="bold")
plt.xlabel("Time Step (t)")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

## 2. Stationarity & Order-1 Differencing
A time series is weakly stationary if its mean, variance, and autocovariance are invariant over time. Differencing removes trends.

In [ ]:
diff_1 = series[1:] - series[:-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(series, color="#1f77b4")
axes[0].set_title("Non-Stationary Series (Upward Trend)", fontweight="bold")
axes[0].set_xlabel("Time")

axes[1].plot(diff_1, color="#2ca02c")
axes[1].axhline(0, color="black", linestyle=":")
axes[1].set_title("Differenced Series y'_t (Stationary Zero-Mean)", fontweight="bold")
axes[1].set_xlabel("Time")
plt.tight_layout()
plt.show()

## 3. Tabular ML Feature Engineering (Lags & Rolling Statistics)
Converting sequential time series into supervised learning matrices $[X_{t-\text{lags}}, Y_t]$ to train GBDT models.

In [ ]:
# Build feature matrix: Lags t-1, t-2, t-12 (seasonal), plus rolling 3-step mean
lags = [1, 2, 12]
X_list, y_list = [], []
for i in range(12, len(series)):
    feat = [series[i - l] for l in lags]
    feat.append(np.mean(series[i - 3 : i]))  # Rolling mean
    X_list.append(feat)
    y_list.append(series[i])

X_tab = np.array(X_list)
y_tab = np.array(y_list)

split_idx = 84
X_tr, y_tr = X_tab[:split_idx], y_tab[:split_idx]
X_te, y_te = X_tab[split_idx:], y_tab[split_idx:]

gbr = GradientBoostingRegressor(n_estimators=80, max_depth=3, random_state=42).fit(X_tr, y_tr)
y_pred = gbr.predict(X_te)

rmse = root_mean_squared_error(y_te, y_pred)
mae = mean_absolute_error(y_te, y_pred)

plt.figure(figsize=(9, 4.5))
test_steps = np.arange(split_idx, len(y_tab))
plt.plot(test_steps, y_te, label="Actual Test Observations", color="black", lw=2)
plt.plot(test_steps, y_pred, label=f"GBDT Forecast (RMSE={rmse:.2f}, MAE={mae:.2f})", color="#d62728", marker="o", linestyle="--")
plt.title("Gradient Boosted Decision Tree Forecast on Tabular Lags", fontweight="bold")
plt.xlabel("Sample Index")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()